In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import json
import os
import random
import math


In [2]:

print("--- 1. RESTORING PROJECT STATE ---")

# A. Load the Data Splits
# We need val_df to know which user bought what
print("Loading validation data...")
if os.path.exists('../data/100k/val_final_mini.parquet'):
    val_df = pd.read_parquet('../data/100k/val_final_mini.parquet')
    print(f"Validation Set: {len(val_df)} rows")
else:
    print("❌ Error: Validation data not found.")

# B. Load the Item Metadata & Embeddings
print("Loading item artifacts...")
all_items_df = pd.read_parquet('../data/100k/all_items_processed_100k.parquet')
item_embeddings = np.load('../data/100k/item_embeddings_100k.npy')

# C. Load the Model
print("Loading trained model...")
model = tf.keras.models.load_model('../data/100k/two_tower_model_100k.keras')
user_emb_layer = model.get_layer('user_emb')

# D. Create Fast Lookup Maps
# Map Item ID -> Index in the embedding matrix
print("Building lookup maps...")
item_id_to_index = pd.Series(all_items_df.index, index=all_items_df['item_id']).to_dict()
all_item_ids = list(all_items_df['item_id'].unique())

print("\n✅ Setup Complete! Ready for evaluation.")

--- 1. RESTORING PROJECT STATE ---
Loading validation data...
Validation Set: 10000 rows
Loading item artifacts...
Loading trained model...
Building lookup maps...

✅ Setup Complete! Ready for evaluation.


In [3]:
# --- CONFIGURATION ---
TOP_K = 10
NUM_NEGATIVES = 50   # 1 Positive vs 50 Negatives
SAMPLE_SIZE = 1000   # Number of users to test (Set to len(val_df) for full run)

print(f"\n--- 2. STARTING EVALUATION (@K={TOP_K}) ---")
print(f"Testing on {SAMPLE_SIZE} users with {NUM_NEGATIVES} negative samples each.")

metrics = {
    'hits': 0,        # For Hit Rate / Recall
    'mrr': 0,         # Mean Reciprocal Rank
    'ndcg': 0,        # Normalized Discounted Cumulative Gain
    'precision': 0,   # Precision
    'count': 0        # Total processed
}

# Sample random users from validation set
eval_data = val_df.sample(n=SAMPLE_SIZE, random_state=42)

for idx, row in eval_data.iterrows():
    # 1. Get User Vector
    uid = int(row['user_id'])
    # Handle Cold Start (Unknown Users -> ID 0)
    if uid >= user_emb_layer.input_dim: 
        uid = 0
    
    # Extract user vector from the model layer
    user_vec = user_emb_layer(np.array([uid])).numpy().reshape(32)
    
    # 2. Get Positive Item Vector (The one they actually bought)
    iid = int(row['item_id'])
    if iid not in item_id_to_index: 
        continue # Skip items missing from our catalog
        
    pos_idx = item_id_to_index[iid]
    pos_vec = item_embeddings[pos_idx]
    
    # 3. Get Negative Vectors (Random stuff they didn't buy)
    neg_vectors = []
    while len(neg_vectors) < NUM_NEGATIVES:
        rand_id = random.choice(all_item_ids)
        # Ensure we don't accidentally pick the true item
        if rand_id != iid and rand_id in item_id_to_index:
            neg_idx = item_id_to_index[rand_id]
            neg_vectors.append(item_embeddings[neg_idx])
            
    # 4. Calculate Scores
    # Score = Dot Product of User Vector * Item Vector
    pos_score = np.dot(user_vec, pos_vec)
    neg_scores = np.dot(neg_vectors, user_vec) 
    
    # 5. Ranking Logic
    # Append positive score to the END of the negative scores list
    all_scores = np.append(neg_scores, pos_score)
    
    # Sort descending to find rank
    # The positive item is at the last index (index = NUM_NEGATIVES)
    sorted_indices = np.argsort(all_scores)[::-1]
    
    # Find where the positive item ended up (Rank 1 = Top)
    rank = np.where(sorted_indices == NUM_NEGATIVES)[0][0] + 1
    
    # 6. Compute Metrics
    metrics['count'] += 1
    
    # MRR (Add 1/rank)
    metrics['mrr'] += 1.0 / rank
    
    # Top-K Metrics
    if rank <= TOP_K:
        # Hit Rate & Recall (Identical in this scenario)
        metrics['hits'] += 1
        
        # NDCG (1 / log2(rank + 1))
        metrics['ndcg'] += 1.0 / math.log2(rank + 1)
        
        # Precision (1/K if hit)
        metrics['precision'] += 1.0 / TOP_K

    # Progress Log
    if metrics['count'] % 200 == 0:
        print(f"   Evaluated {metrics['count']} users...")

# --- FINAL RESULTS ---
n = metrics['count']
results = {
    f'Hit Rate@{TOP_K}': metrics['hits'] / n,
    f'Recall@{TOP_K}':   metrics['hits'] / n,
    f'Precision@{TOP_K}': metrics['precision'] / n,
    f'NDCG@{TOP_K}':     metrics['ndcg'] / n,
    'MRR':              metrics['mrr'] / n
}

print("\n" + "="*40)
print(f"   🏆 FINAL RESULTS (N={n})")
print("="*40)
for name, value in results.items():
    print(f"{name:<20} : {value:.4f}")
print("="*40)


--- 2. STARTING EVALUATION (@K=10) ---
Testing on 1000 users with 50 negative samples each.
   Evaluated 200 users...
   Evaluated 400 users...
   Evaluated 600 users...
   Evaluated 800 users...
   Evaluated 1000 users...

   🏆 FINAL RESULTS (N=1000)
Hit Rate@10          : 0.3240
Recall@10            : 0.3240
Precision@10         : 0.0324
NDCG@10              : 0.1451
MRR                  : 0.1176
